In [1]:
import os
import psycopg
from dotenv import load_dotenv
#Random
load_dotenv()

csv_file = "data\mockup_data.csv"

conn = psycopg.connect(
    f"dbname={os.getenv('POSTGRES_DB')} user={os.getenv('POSTGRES_USER')} password={os.getenv('POSTGRES_PASSWORD')} host={os.getenv('POSTGRES_HOST')}"
)

with conn.cursor() as cur:


    cur.execute("""
        CREATE SCHEMA IF NOT EXISTS raw
    """)


    cur.execute("""
        CREATE TABLE IF NOT EXISTS raw.csv_raw (
            line TEXT
        )
    """)

    conn.commit()


    with open(csv_file, "r", encoding="utf-8", errors="surrogateescape") as f:
        with cur.copy("COPY raw.csv_raw (line) FROM STDIN") as copy:
            for line in f:
                copy.write_row([line.rstrip("\n")])

conn.commit()
conn.close()

<>:7: SyntaxWarning: invalid escape sequence '\m'
<>:7: SyntaxWarning: invalid escape sequence '\m'
C:\Users\kalle\AppData\Local\Temp\ipykernel_19596\3004256689.py:7: SyntaxWarning: invalid escape sequence '\m'
  csv_file = "data\mockup_data.csv"
C:\Users\kalle\AppData\Local\Temp\ipykernel_19596\3004256689.py:7: SyntaxWarning: invalid escape sequence '\m'
  csv_file = "data\mockup_data.csv"


ProgrammingError: missing "=" after "#" in connection info string


In [ ]:
import os
import psycopg
import pandas as pd
from dotenv import load_dotenv
from io import StringIO

load_dotenv()

conn = psycopg.connect(
    f"dbname={os.getenv('POSTGRES_DB')} user={os.getenv('POSTGRES_USER')} password={os.getenv('POSTGRES_PASSWORD')} host={os.getenv('POSTGRES_HOST')}"
)

lines = []

with conn.cursor() as cur:
    cur.execute("SELECT line FROM raw.csv_raw")

    for row in cur:
        lines.append(row[0])

conn.close()

csv_text = "\n".join(lines)
csv_buffer = StringIO(csv_text)

df = pd.read_csv(csv_buffer, sep=';')
df

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['temp'] = pd.to_numeric(df['temp'], errors='coerce')

df

In [ ]:
reject_condition = (
       (df['created_at'].isna()) |
       (df['temp'].isna())
)

df_rejected = df[reject_condition].copy()
df_valid = df[~reject_condition].copy()

df_rejected.to_csv('rejected_products.csv',sep=',', index=False)
df_valid.to_csv('accepted_products.csv', sep=',', index=False)

In [ ]:
import os
import psycopg
import pandas as pd
from io import StringIO


df_clean = pd.read_csv('accepted_products.csv')

conn = psycopg.connect(
    f"dbname={os.getenv('POSTGRES_DB')} user={os.getenv('POSTGRES_USER')} password={os.getenv('POSTGRES_PASSWORD')} host={os.getenv('POSTGRES_HOST')}"
)
with conn.cursor() as cur:


    cur.execute("CREATE SCHEMA IF NOT EXISTS clean")


    cur.execute("""
        CREATE TABLE IF NOT EXISTS clean.weather_clean (
            id SERIAL PRIMARY KEY,
            created_at DATE,
            temp FLOAT
        )
    """)

    conn.commit()


    buffer = StringIO()
    df_clean.to_csv(buffer, index=False, header=False)
    buffer.seek(0)

    with cur.copy("COPY clean.weather_clean (created_at, temp) FROM STDIN WITH CSV") as copy:
        for line in buffer:
            copy.write(line)

conn.commit()
conn.close()

In [ ]:
import os
import psycopg
import pandas as pd
from datetime import datetime


user_input = input("Enter a date (YYYY-MM-DD): ").strip()


try:
    input_date = datetime.strptime(user_input, "%Y-%m-%d").date()
except ValueError:
    print("Invalid date format. Please use YYYY-MM-DD.")
    exit(1)


conn = psycopg.connect(
    f"dbname={os.getenv('POSTGRES_DB')} user={os.getenv('POSTGRES_USER')} password={os.getenv('POSTGRES_PASSWORD')} host={os.getenv('POSTGRES_HOST')}"
)


query = """
SELECT temp
FROM clean.weather_clean
WHERE DATE(created_at) = %s
"""

with conn.cursor() as cur:
    cur.execute(query, (input_date,))
    rows = cur.fetchall()
    colnames = [desc[0] for desc in cur.description]

conn.close()


df = pd.DataFrame(rows, columns=colnames)

if df.empty:
    print(f"No data found for {input_date}")
else:
    print(f"Data for {input_date}:")
    print(df)